In [2]:
import torch 
from torch import nn
from torch.nn import functional as F

In [5]:
net=nn.Sequential(nn.Linear(20,256),nn.ReLU(),nn.Linear(256,10))

x=torch.rand(2,20)
net(x)

tensor([[-0.0531, -0.0650,  0.1633,  0.2136, -0.1406,  0.0429, -0.1590,  0.1473,
          0.0510,  0.1254],
        [-0.0176, -0.1094,  0.1823,  0.1232, -0.1440, -0.0094, -0.2388,  0.0638,
          0.1300,  0.0996]], grad_fn=<AddmmBackward0>)

In [7]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden=nn.Linear(20,256)
        self.out=nn.Linear(256,10)

    def forward(self,x):
        return self.out(F.relu(self.hidden(x)))

In [8]:
net=MLP()
net(x)

tensor([[-0.2043,  0.0162, -0.0464,  0.2695,  0.1391,  0.2589,  0.0628,  0.1179,
         -0.1428,  0.0323],
        [-0.0859,  0.0224, -0.0575,  0.1957,  0.2754,  0.1861,  0.1087,  0.1692,
         -0.1372,  0.1361]], grad_fn=<AddmmBackward0>)

MySequential 中，*args 的作用就是让你可以一次传入任意数量的网络层，而不必提前规定网络必须有几层。

In [10]:
class MySequential(nn.Module):
    def __init__(self,*args):
        super().__init__()
        for block in args:
            self._modules[block]=block
    def forward(self,x):
        for block in self._modules.values():
            x=block(x)
        return x

net =MySequential(
    nn.Linear(20,256),
    nn.ReLU(),
    nn.Linear(256,10)
)
net(x)

tensor([[ 0.1110,  0.0343,  0.1448, -0.0283,  0.0575, -0.2468, -0.1548, -0.0335,
          0.0035, -0.1902],
        [ 0.2592,  0.0387,  0.1748,  0.1235,  0.0267, -0.3451, -0.1572, -0.1746,
          0.0783, -0.1499]], grad_fn=<AddmmBackward0>)

In [12]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.rand_weight=torch.rand((10,20),requires_grad=False)
        self.linear=nn.Linear(20,10)

    def forward(self,x):
        x=self.linear(x)
        x=F.relu(torch.mm(x,self.rand_weight))
        return x.sum()

net=FixedHiddenMLP()
net(x)    

tensor(8.2132, grad_fn=<SumBackward0>)

In [16]:
class NextMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(20,64),
            nn.ReLU(),
            nn.Linear(64,128),
            nn.ReLU()
        )
        self.linear=nn.Linear(128,2)

    def forward(self,x):
        return self.linear(self.net(x))

chimear=nn.Sequential(
    NextMLP(),
    nn.Linear(2,20),
    FixedHiddenMLP()
)
chimear(x)


tensor(1.0608, grad_fn=<SumBackward0>)